# 05 — Feature Selection

**Objective:** Plan leakage-safe feature-selection experiments.  
**Owner:** TBD  
**Sprint:** TBD  

> Leakage warning: select features independently inside each training fold.

In [ ]:
from pathlib import Path

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "configs" / "config.yaml").is_file())
import sys
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from src.config import load_config

config = load_config()
print(config["project"]["name"])
print(config["project"]["random_state"])

## Planned work

Later: define selection methods, nest them in validated pipelines, compare feature counts, and document stability and interpretation.

In [ ]:
from src.data import load_dataset

X, y, metadata = load_dataset(optimize_memory=True)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression

feature_selector = SelectFromModel(
    LogisticRegression(penalty="l1", solver="saga", C=0.1, random_state=config["project"]["random_state"])
)
classifier = LogisticRegression(
    penalty="l2", solver="lbfgs", C=1.0, max_iter=1000, random_state=config["project"]["random_state"])

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("selector", feature_selector),
    ("classifier", classifier)
])

In [ ]:
from src.evaluation import evaluate_model_cv

fold_results, summary = evaluate_model_cv(
    estimator=pipeline,
    X=X,
    y=y,
    model_name="L1-FeatureSelection + LogisticRegression",
    experiment_id="M03-FS-001",
    member="Member 03",
    branch="feature/feature-selection"
)
print(f"Primary Metric ({summary['primary_metric']}): {summary['primary_score_mean']:.4f} +/- {summary['primary_score_std']:.4f}")